In [ ]:
%reload_ext autoreload
%autoreload 2

# Agent-Based Task Execution

This notebook demonstrates how to use the `Agent` pipeline from **[OnPrem.LLM](https://github.com/amaiya/onprem)** to create autonomous agents that can execute complex tasks using a variety of tools.

## The AgentExecutor

The `AgentExecutor` allows you to launch AI agents to solve various tasks using both cloud and local models. We will use **gpt-5-mini** (cloud) and **glm-4.7-flash** (local) for these examples.

By default, the `AgentExecutor` has access to 9 built-in tools.  You remove access to built-in-tools as necessary.  You can optionally give the agent access to [custom tools](https://amaiya.github.io/patchpal/features/custom-tools/) by placing them in `~/.patchpal/tools`.

The `AgentExecutor` is implemented using our coding agent, PatchPal, which you'll need to install: `pip install patchpal`.

In [ ]:
# | notest

from onprem.pipelines import AgentExecutor

In [ ]:
# | notest

AgentExecutor.print_default_tools()

AgentExecutor Default Tools

These tools are used by default when enabled_tools=None:

   1. read_file       - Read complete file contents
   2. read_lines      - Read specific line ranges from files
   3. edit_file       - Edit files via find/replace
   4. write_file      - Write complete file contents
   5. grep            - Search for patterns in files
   6. find            - Find files by glob pattern
   7. run_shell       - Execute shell commands
   8. web_search      - Search the web for information
   9. web_fetch       - Fetch and read content from URLs

Customization Examples:

# Use defaults (all tools including shell):
executor = AgentExecutor(model='anthropic/claude-sonnet-4-5')

# Defaults but no shell access (safer):
executor = AgentExecutor(
    model='openai/gpt-5-mini',
    disable_shell=True
)

# Minimal tools:
executor = AgentExecutor(
    model='openai/gpt-5-mini',
    enabled_tools=['read_file', 'write_file']
)

# Web research only:
executor = AgentExecutor(
    mo

## Examples

Let's run through some examples for different scenarios.

### Basic Calculator Example

In this introductory example, we will launch an agent to build a **calculator module in Python**.

By default, the agent will operate within the working_directory you specify (or the current folder if not working directory is specified).  The agent cannot read or write outside the working directory.

**Important Note:**:  If the agent has access to the `run_shell` tool, it can potentially read or modify files outside of the working directory (e.g., auto-generating and executing a Python script that writes files outside of working directory).  For these reasons, you can either supply the `disable_shell=True` to remove shell access or `sandbox=True`, which runs the agent in an ephemeral container.  We will cover sandboxed execution later. 

We will run this example without either, as we are running this notebook within a VM.

In [ ]:
# | notest

from onprem.pipelines.agent import AgentExecutor

executor = AgentExecutor(
     model='openai/gpt-5-mini', # assumes OPENAI_API_KEY is already set as environment variable
     max_iterations=10
 )

result = executor.run(
     task="""
     Create a simple Python calculator module with the following:
     - calculator.py with add, subtract, multiply, divide functions
     - test_calculator.py with pytest tests
     - All tests must pass
     """,
     working_dir='./calculator_project'
)
print(result)


✈️  PatchPal Autopilot Mode Starting
Prompt: 
     Create a simple Python calculator module with the following:
     - calculator.py with add, su...
Completion promise: 'COMPLETE'
Max iterations: 20
Model: openai/gpt-5-mini
Working directory: /home/amaiya/projects/ghub/onprem/nbs/calculator_project
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/20

🤔 Thinking...

I'll create two files: calculator.py with the four functions, and               
test_calculator.py with pytest tests (including a division-by-zero test). Then  
I'll run the test suite to ensure all tests pass. I'll write the files now.     

📝 Patching: calculator.py
🤔 Thinking...
📝 Patching: test_calculator.py
🤔 Thinking...

I'll run pytest to verify all tests pass.                                       

⚡ Running: pytest -q
🤔 Thinking...

📝 Agent Response:
<promise>COMPLETE</promise>

💰 Cumulative Cost (after iteration 1):
   Total LLM calls: 4
   Total tokens: 10,548
   Total cost: $0.0031


✅ C

### Web Research Agent

This example is a web research agent that only has access to the following tools: `web_search`, `web_fetch`, `write_file`.

In [ ]:
# | notest

prompt = """
Research the latest developments in quantum computing in 2026.
Create a markdown report called 'quantum_computing_2026.md' with:
- Executive summary
- Key breakthroughs
- Major companies/institutions involved
- Potential applications
- Sources and references
"""

In [ ]:
# | notest
from onprem.pipelines.agent import AgentExecutor

executor = AgentExecutor(
     model='openai/gpt-5-mini',
     max_iterations=10,
    enabled_tools=["web_search", "web_fetch", "write_file"],
 )

result = executor.run(
     task=prompt,
     working_dir='./quantum_report'
)


✈️  PatchPal Autopilot Mode Starting
Prompt: 
Research the latest developments in quantum computing in 2026.
Create a markdown report called 'qua...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: openai/gpt-5-mini
Working directory: /home/amaiya/projects/ghub/onprem/nbs/quantum_report
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...

I'll search recent news and articles about quantum computing developments in    
2026, then compile a markdown report. I'll pull authoritative sources (company  
blogs, academic papers, news outlets, and preprints).                           

🌐 Searching web: quantum computing 2026 breakthroughs 2026 news
🤔 Thinking...
🌐 Searching web: 2026 quantum computing major developments 2026 'quantum advantage' 2026 'logical qubit' 2026 IBM 2026 Google 'IonQ' 'Quantinuum' 'PsiQuantum'
🤔 Thinking...
🌐 Searching web: IBM March 12 2026 blueprint quantum-centric supercomputing IBM 2026 press release 'March 12 2026' 'q

## Local Models

The `AgentExecutor` supports local models.  By default, it will assume the local model supports native function-calling (e.g., gpt-oss-120b).  If you use a local model that does **not** have good native support for function-calling (a.k.a. tool-calling), you can change the agent_type to `react`. In this example, we will use `llama3.1:8b`.

**Note:** The default context window length in Ollama is typically too small for agentic  workflows.  Depending on the model and task, we recommend inreasing to at least 8192.  Reasoning models like gpt-oss:120b may require 32K or 64K.  

```sh
OLLAMA_CONTEXT_LENGTH=32000 ollama serve
```

In [ ]:
# | notest

from onprem.pipelines.agent import AgentExecutor

executor = AgentExecutor(
     model='ollama_chat/qwen3:8b',
     enabled_tools=["web_fetch", "write_file"],
     max_iterations=10
 )

result = executor.run(
     task="What is the highest level of education of the person listed on this page: https://arun.maiya.net? Write answer in answer.txt.",
     working_dir='./extraction_example'
)


✈️  PatchPal Autopilot Mode Starting
Prompt: What is the highest level of education of the person listed on this page: https://arun.maiya.net? Wr...
Completion promise: 'COMPLETE'
Max iterations: 10
Model: ollama_chat/qwen3:8b
Working directory: /home/amaiya/projects/ghub/onprem/nbs/extraction_example
🔒 File access restricted to working directory


🔄 Autopilot Iteration 1/10

🤔 Thinking...
🌐 Fetching: https://arun.maiya.net
🤔 Thinking...
🌐 Fetching: https://arun.maiya.net
🤔 Thinking...
